In [ ]:
# ========================================
#  环境初始化：激活项目环境
# ========================================
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

In [ ]:
# ========================================
#  导入依赖：LinearAlgebra, ForwardDiff, PyPlot
# ========================================
using LinearAlgebra
using ForwardDiff
using PyPlot

In [ ]:
# ========================================
#  单摆动力学方程 (同 L2)
#  返回: [θ̇, θ̈]
# ========================================
function pendulum_dynamics(x)
    l = 1.0
    g = 9.81
    
    θ = x[1]
    θ̇ = x[2]
    
    θ̈ = -(g/l)*sin(θ)
    
    return [θ̇; θ̈]
end

In [ ]:
# ========================================
#  后向 Euler 单步——用不动点迭代求解隐式方程
#  原理: 重复 x_{k+1} ← x_k + h·f(x_{k+1}) 直到收敛
#  返回: 收敛后的状态和误差历史
# ========================================
function backward_euler_step_fixed_point(fun, x0, h)
    xn = x0
    e = [norm(x0 + h.*fun(xn) - xn)]
    while e[end] > 1e-8
        xn = x0 + h.*fun(xn)
        e = [e; norm(x0 + h.*fun(xn) - xn)]
    end
    
    return xn, e
end

In [ ]:
# ========================================
#  后向 Euler 单步——用 Newton 法求解隐式方程
#  残差: r(x_{k+1}) = x_k + h·f(x_{k+1}) - x_{k+1} = 0
#  用 Newton 法求根，每步 1-2 次迭代即可收敛
#  比不动点迭代快得多
# ========================================
function backward_euler_step_newton(fun, x0, h)
    xn = x0
    r = x0 + h.*fun(xn) - xn
    e = [norm(r)]
    while e[end] > 1e-8
        ∂r = ForwardDiff.jacobian(x -> x0 + h.*fun(x) - x, xn)
        xn = xn - ∂r\r
        r = x0 + h.*fun(xn) - xn
        e = [e; norm(r)]
    end
    
    return xn, e
end

In [ ]:
# ========================================
#  后向 Euler 完整仿真 (不动点迭代版)
# ========================================
function backward_euler_fixed_point(fun, x0, Tf, h)
    t = Array(range(0,Tf,step=h))
    
    x_hist = zeros(length(x0),length(t))
    x_hist[:,1] .= x0
    
    for k = 1:(length(t)-1)
        x_hist[:,k+1], e = backward_euler_step_fixed_point(fun, x_hist[:,k], h)
    end
    
    return x_hist, t
end

In [ ]:
# ========================================
#  后向 Euler 完整仿真 (Newton 版)
# ========================================
function backward_euler_newton(fun, x0, Tf, h)
    t = Array(range(0,Tf,step=h))
    
    x_hist = zeros(length(x0),length(t))
    x_hist[:,1] .= x0
    
    for k = 1:(length(t)-1)
        x_hist[:,k+1], e = backward_euler_step_newton(fun, x_hist[:,k], h)
    end
    
    return x_hist, t
end

In [ ]:
# ========================================
#  测试：两种后向 Euler 方法的结果差异
# ========================================
x0 = [.1; 0]
x_hist1, t_hist1 = backward_euler_fixed_point(pendulum_dynamics, x0, 10, 0.01)
x_hist2, t_hist2 = backward_euler_newton(pendulum_dynamics, x0, 10, 0.01)
plot(t_hist1, x_hist1[1,:])
plot(t_hist2, x_hist2[1,:])

In [ ]:
# ========================================
#  验证两种方法结果一致
# ========================================
max(abs.(x_hist1-x_hist2)...)

In [ ]:
# ========================================
#  测试单步收敛行为
# ========================================
xn, e1 = backward_euler_step_fixed_point(pendulum_dynamics, x0, 0.1)
e1

In [ ]:
# ========================================
#  比较 Newton 法
# ========================================
xn, e2 = backward_euler_step_newton(pendulum_dynamics, x0, 0.1)
e2

In [ ]:
# ========================================
#  绘制误差收敛曲线 (对数坐标)
# ========================================
semilogy(e1)
semilogy(e2)